# **Ajuste Fino de Hiperparâmetros — Multi-Dataset**

Busca aleatória com **foco em minimizar o número total de parâmetros** do modelo,
mantendo boa acurácia. O código se adapta automaticamente ao dataset escolhido.

---

### Datasets de classificação de imagens disponíveis no Keras

| String            | Dataset       | Imagem  | Canais    | Classes |
|-------------------|---------------|---------|-----------|--------|
| `"mnist"`         | MNIST Dígitos | 28 × 28 | 1 (cinza) | 10     |
| `"fashion_mnist"` | Fashion-MNIST | 28 × 28 | 1 (cinza) | 10     |
| `"cifar10"`       | CIFAR-10      | 32 × 32 | 3 (RGB)   | 10     |
| `"cifar100"`      | CIFAR-100     | 32 × 32 | 3 (RGB)   | 100    |

## ⚙️ CONFIGURAÇÃO — altere apenas aqui

In [1]:
# ════════════════════════════════════════════════════════════════
#  🔧 MUDE APENAS ESTA STRING PARA TROCAR O DATASET
#     opções: "mnist" | "fashion_mnist" | "cifar10" | "cifar100"
DATASET_NAME = "fashion_mnist"
# ════════════════════════════════════════════════════════════════

# Busca aleatória
NUM_TRIALS     = 10   # quantas combinações de hiperparâmetros testar
EXECUTIONS     = 1    # repetições por trial (1 = mais rápido)
EPOCHS_SEARCH  = 4    # épocas por trial (rápido)

# Treino final
EPOCHS_FINAL   = 20   # máx. de épocas (EarlyStopping interrompe antes se necessário)

# Geral
BATCH_SIZE     = 128  # batch maior → treino mais rápido
VAL_SIZE       = 5000 # amostras reservadas para validação
SEED           = 42

# Peso da penalização de tamanho no objetivo composto:
#   score = val_accuracy − LAMBDA × log10(num_params)
# ↑ LAMBDA → favorece modelos menores  |  ↓ LAMBDA → favorece acurácia
LAMBDA         = 0.02

## Instalações e imports

In [2]:
!pip install -q keras-tuner~=1.4.6

import sys, shutil, math
import numpy as np
import tensorflow as tf
import keras_tuner as kt

from tensorflow import keras
from tensorflow.keras import layers

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow {tf.__version__}  |  Python {sys.version.split()[0]}")

TensorFlow 2.19.0  |  Python 3.12.13


## Carregamento e preparo dos dados

In [3]:
# ── Mapa completo: todos os datasets de classificação de imagens do Keras ─────
DATASET_MAP = {
    "mnist":         {"loader": keras.datasets.mnist.load_data,         "num_classes": 10},
    "fashion_mnist": {"loader": keras.datasets.fashion_mnist.load_data,  "num_classes": 10},
    "cifar10":       {"loader": keras.datasets.cifar10.load_data,        "num_classes": 10},
    "cifar100":      {"loader": keras.datasets.cifar100.load_data,       "num_classes": 100},
}

# Validação antecipada — mensagem clara se a string estiver errada
assert DATASET_NAME in DATASET_MAP, (
    f"Dataset '{DATASET_NAME}' inválido.\n"
    f"Opções válidas: {list(DATASET_MAP.keys())}"
)

cfg         = DATASET_MAP[DATASET_NAME]
NUM_CLASSES = cfg["num_classes"]

# ── Carrega ───────────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = cfg["loader"]()

# CIFAR retorna labels com shape (N,1) — achatar para (N,)
y_train = y_train.reshape(-1)
y_test  = y_test.reshape(-1)

# ── Normalização 0–255 → 0.0–1.0 ─────────────────────────────────────────────
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

# ── Adaptação automática de canal ─────────────────────────────────────────────
# Escala de cinza (H,W) → (H,W,1) | RGB (H,W,3) fica igual
if x_train.ndim == 3:
    x_train = np.expand_dims(x_train, axis=-1)
    x_test  = np.expand_dims(x_test,  axis=-1)

INPUT_SHAPE = x_train.shape[1:]  # (H, W, C) — detectado automaticamente

# ── Separação treino / validação ──────────────────────────────────────────────
x_val,   x_train = x_train[:VAL_SIZE], x_train[VAL_SIZE:]
y_val,   y_train = y_train[:VAL_SIZE], y_train[VAL_SIZE:]

print(f"  Dataset  : {DATASET_NAME}")
print(f"  x_train  : {x_train.shape}")
print(f"  x_val    : {x_val.shape}")
print(f"  x_test   : {x_test.shape}")
print(f"  Classes  : {NUM_CLASSES}")
print(f"  Input    : {INPUT_SHAPE}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
  Dataset  : fashion_mnist
  x_train  : (55000, 28, 28, 1)
  x_val    : (5000, 28, 28, 1)
  x_test   : (10000, 28, 28, 1)
  Classes  : 10
  Input    : (28, 28, 1)


## Definição do modelo e espaço de busca

### Por que o espaço é compacto?

O objetivo é encontrar o **menor número total de parâmetros** que ainda produza boa acurácia.
Um espaço de busca grande gera modelos grandes desnecessariamente e torna a busca lenta.

Hiperparâmetros buscados:

| HP             | Opções              | Impacto nos parâmetros |
|----------------|---------------------|------------------------|
| `n_conv_blocks`| 1 ou 2              | Alto — dobrar blocos ≈ dobrar parâmetros conv |
| `filters`      | 16 / 32 / 64        | Alto — quadrático na camada flatten            |
| `dense_units`  | 64 / 128            | Médio                                          |
| `dropout`      | True / False        | Nenhum — não gera parâmetros                   |
| `lr`           | 1e-3 → 8e-3 (log)   | Nenhum                                         |

In [4]:
def build_model(hp):
    """CNN compacta adaptável a qualquer INPUT_SHAPE e NUM_CLASSES."""

    # ── Hiperparâmetros ───────────────────────────────────────────────────────
    n_conv  = hp.Int   ("n_conv_blocks", min_value=1, max_value=2)
    filters = hp.Choice("filters",       [16, 32, 64])
    dense   = hp.Choice("dense_units",   [64, 128])
    lr      = hp.Float ("lr", 1e-3, 8e-3, sampling="log")
    dropout = hp.Boolean("dropout")

    # ── Arquitetura ───────────────────────────────────────────────────────────
    inputs = keras.Input(shape=INPUT_SHAPE)
    x = inputs

    for _ in range(n_conv):
        x = layers.Conv2D(filters, kernel_size=3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)

    if dropout:
        x = layers.Dropout(0.3)(x)

    x = layers.Dense(dense, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## RandomSearch — busca focada em modelos compactos

In [5]:
# Limpa busca anterior do mesmo dataset
tuner_dir = f"tuner_{DATASET_NAME}"
shutil.rmtree(tuner_dir, ignore_errors=True)

tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=NUM_TRIALS,
    executions_per_trial=EXECUTIONS,
    directory=tuner_dir,
    project_name="search",
    seed=SEED,
)

print(tuner.search_space_summary())

Search space summary
Default search space size: 5
n_conv_blocks (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 2, 'step': 1, 'sampling': 'linear'}
filters (Choice)
{'default': 16, 'conditions': [], 'values': [16, 32, 64], 'ordered': True}
dense_units (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128], 'ordered': True}
lr (Float)
{'default': 0.001, 'conditions': [], 'min_value': 0.001, 'max_value': 0.008, 'step': None, 'sampling': 'log'}
dropout (Boolean)
{'default': False, 'conditions': []}
None


In [6]:
early_search = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=2, restore_best_weights=True
)

tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_SEARCH,
    batch_size=BATCH_SIZE,
    callbacks=[early_search],
    verbose=1,
)

Trial 10 Complete [00h 01m 13s]
val_accuracy: 0.9075999855995178

Best val_accuracy So Far: 0.9200000166893005
Total elapsed time: 00h 19m 39s


## Relatório — todos os trials ordenados por menor nº de parâmetros

In [7]:
trials  = tuner.oracle.trials
results = []

for trial_id, trial in trials.items():
    hp_vals    = trial.hyperparameters.values
    val_acc    = trial.score
    model_tmp  = build_model(trial.hyperparameters)
    num_params = model_tmp.count_params()
    keras.backend.clear_session()  # libera memória dos modelos temporários

    row = hp_vals.copy()
    row["val_accuracy"] = val_acc
    row["num_params"]   = num_params

    # Score composto penalizado: favorece modelos com menos parâmetros
    if val_acc is not None:
        row["score_composto"] = val_acc - LAMBDA * math.log10(max(num_params, 1))
    else:
        row["score_composto"] = None

    results.append(row)

# Ordenar por num_params ASC (menor primeiro)
results.sort(key=lambda r: r.get("num_params", float("inf")))

COLS = ["n_conv_blocks", "filters", "dense_units", "lr",
        "dropout", "num_params", "val_accuracy", "score_composto"]
W = 15

header = " | ".join(f"{k:{W}s}" for k in COLS)
sep    = "-" * len(header)

print(f"\n{'='*len(header)}")
print(f"  {DATASET_NAME.upper()} — trials ordenados por MENOR nº de parâmetros")
print(f"  score_composto = val_accuracy − {LAMBDA} × log10(num_params)")
print(f"{'='*len(header)}")
print(header)
print(sep)

for r in results:
    row_vals = []
    for k in COLS:
        v = r.get(k, "-")
        if k in ("val_accuracy", "score_composto") and isinstance(v, float):
            v = f"{v*100:.3f}%" if k == "val_accuracy" else f"{v:.4f}"
        elif k == "num_params" and isinstance(v, int):
            v = f"{v:,}"
        elif isinstance(v, float):
            v = f"{v:.5f}"
        row_vals.append(f"{str(v):{W}s}")
    print(" | ".join(row_vals))

print(sep)


  FASHION_MNIST — trials ordenados por MENOR nº de parâmetros
  score_composto = val_accuracy − 0.02 × log10(num_params)
n_conv_blocks   | filters         | dense_units     | lr              | dropout         | num_params      | val_accuracy    | score_composto 
---------------------------------------------------------------------------------------------------------------------------------------------
2               | 16              | 64              | 0.00176         | True            | 53,370          | 90.080%         | 0.8063         
2               | 32              | 64              | 0.00237         | False           | 110,634         | 90.500%         | 0.8041         
2               | 32              | 128             | 0.00121         | True            | 211,690         | 90.480%         | 0.7983         
2               | 64              | 64              | 0.00176         | True            | 238,986         | 92.000%         | 0.8124         
1               | 32      

## Melhor modelo — hiperparâmetros e treino final

In [8]:
best_hp    = tuner.get_best_hyperparameters(1)[0]
best_model = build_model(best_hp)

print("\n" + "="*44)
print("   MELHORES HIPERPARÂMETROS")
print("="*44)
print(f"  {'PARÂMETRO':<24} VALOR")
print("-"*44)
for key, value in sorted(best_hp.values.items()):
    print(f"  {key:<24} {value}")
print("-"*44)
print(f"  {'Total de parâmetros':<24} {best_model.count_params():,}")
print("="*44)


   MELHORES HIPERPARÂMETROS
  PARÂMETRO                VALOR
--------------------------------------------
  dense_units              64
  dropout                  True
  filters                  64
  lr                       0.0017586560319055038
  n_conv_blocks            2
--------------------------------------------
  Total de parâmetros      238,986


In [9]:
callbacks_final = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, verbose=1
    ),
]

history = best_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_FINAL,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_final,
)

_, test_acc = best_model.evaluate(x_test, y_test, verbose=0)

print("\n" + "="*44)
print(" RESULTADO FINAL")
print("="*44)
print(f"  Dataset              : {DATASET_NAME}")
print(f"  Acurácia (val)       : {max(history.history['val_accuracy'])*100:.2f}%")
print(f"  Acurácia (test)      : {test_acc*100:.2f}%")
print(f"  Parâmetros do modelo : {best_model.count_params():,}")
print("="*44)

Epoch 1/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 68s 155ms/step - accuracy: 0.8293 - loss: 0.4714 - val_accuracy: 0.8886 - val_loss: 0.3135 - learning_rate: 0.0018
Epoch 2/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 65s 152ms/step - accuracy: 0.8867 - loss: 0.3111 - val_accuracy: 0.9008 - val_loss: 0.2708 - learning_rate: 0.0018
Epoch 3/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 64s 148ms/step - accuracy: 0.9011 - loss: 0.2701 - val_accuracy: 0.9102 - val_loss: 0.2472 - learning_rate: 0.0018
Epoch 4/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 82s 149ms/step - accuracy: 0.9101 - loss: 0.2416 - val_accuracy: 0.9160 - val_loss: 0.2304 - learning_rate: 0.0018
Epoch 5/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 66s 154ms/step - accuracy: 0.9169 - loss: 0.2238 - val_accuracy: 0.9216 - val_loss: 0.2165 - learning_rate: 0.0018
Epoch 6/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 65s 150ms/step - accuracy: 0.9235 - loss: 0.2029 - val_accuracy: 0.9238 - val_loss: 0.2118 - learning_rate: 0.0018
Epoch 7/20
430/430 ━━━━━━━━━━━━━━━━━━━━ 81s 149ms/step - accuracy: 0.9